Cleaning Data and Sanity Check

In [1]:
import pandas as pd 

df = pd.read_csv("backend/data/ev_population.csv", low_memory=False)
print(f"Rows: {len(df):,} Columns: {df.shape[1]}")
display(df.head())
df.info()

Rows: 300,142 Columns: 16


,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 GEOID
0,3FMTK3SU8P,King,Seattle,WA,98177.0,2023,FORD,MUSTANG MACH-E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,32.0,254622924,POINT (-122.37238 47.74068),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303300e+10
1,JM3KKEHA8S,Thurston,Yelm,WA,98597.0,2025,MAZDA,CX-90,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,26.0,2.0,277791643,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10
2,5YJ3E1EB8J,King,Sammamish,WA,98075.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,41.0,347418676,POINT (-122.03539 47.61344),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10
3,JHMZC5F12K,Island,Oak Harbor,WA,98277.0,2019,HONDA,CLARITY,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,47.0,10.0,223620701,POINT (-122.64682 48.29077),PUGET SOUND ENERGY INC,5.302997e+10
4,1C4RJXN65R,Kitsap,Bremerton,WA,98312.0,2024,JEEP,WRANGLER,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,21.0,26.0,256402674,POINT (-122.65223 47.57192),PUGET SOUND ENERGY INC,5.303508e+10


<class 'pandas.DataFrame'>
RangeIndex: 300142 entries, 0 to 300141
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         300142 non-null  str    
 1   County                                             300127 non-null  str    
 2   City                                               300127 non-null  str    
 3   State                                              300142 non-null  str    
 4   Postal Code                                        300127 non-null  float64
 5   Model Year                                         300142 non-null  int64  
 6   Make                                               300142 non-null  str    
 7   Model                                              300142 non-null  str    
 8   Electric Vehicle Type                              300142 non-null  str    
 9   Clea

Check for Missing Values

In [2]:
missing = pd.DataFrame({
    "missing": df.isna().sum(), 
    "percent": (df.isna().mean() * 100).round(2)
    }).sort_values("percent", ascending=False)
missing

,missing,percent
Legislative District,790,0.26
Electric Range,35,0.01
Vehicle Location,24,0.01
VIN (1-10),0,0.00
County,15,0.00
City,15,0.00
State,0,0.00
Postal Code,15,0.00
Model Year,0,0.00
Make,0,0.00


Check For Duplicate Rows

In [3]:
print("Fully Duplicate rows:", df.duplicated().sum())
print("Duplicate Vehicle IDs:", df["DOL Vehicle ID"].duplicated().sum())

Fully Duplicate rows: 0
Duplicate Vehicle IDs: 0


Check For the Value Ranges

In [4]:
display(df[["Model Year", "Electric Range"]].describe())
display(df["Model Year"].value_counts().sort_index())

zero = df["Electric Range"] == 0
print(f"Range = 0: {zero.sum():,} rows ({zero.mean() * 100:.1f}%)")

display((zero.groupby(df["Model Year"]).mean() * 100).round(1))

print(df["State"].value_counts().head(), "\n")
print(df["Electric Vehicle Type"].value_counts(), "\n")
print(f"{df['Make'].nunique()} makes")
display(df["Make"].value_counts().head(15))

wa = df[df["State"] == "WA"]
counties = wa["County"].value_counts()
print(f"{counties.size} WA counties")
print(f"Largest county share: {counties.iloc[0] / counties.sum() * 100:.1f}% ({counties.index[0]})")
display(counties.head(10))

multi = df["Electric Utility"].str.contains(r"\|", na=False)
print(f"Rows with several utilities in one field: {multi.sum():,} ({multi.mean() * 100:.1f}%)")

,Model Year,Electric Range
count,300142.000000,300107.000000
mean,2022.389056,36.148437
std,3.095011,75.443498
min,1999.000000,0.000000
25%,2021.000000,0.000000
50%,2023.000000,0.000000
75%,2025.000000,32.000000
max,2027.000000,337.000000


Model Year
1999        3
2000        8
2002        1
2003        1
2008       17
2010       20
2011      521
2012     1266
2013     3772
2014     3101
2015     4341
2016     4992
2017     8353
2018    13745
2019    10666
2020    12016
2021    20685
2022    29963
2023    61487
2024    49111
2025    36561
2026    38472
2027     1040
Name: count, dtype: int64

Range = 0: 197,085 rows (65.7%)


Model Year
1999     0.0
2000     0.0
2002     0.0
2003     0.0
2008     5.9
2010     0.0
2011     0.0
2012     0.0
2013     0.0
2014     0.0
2015     0.0
2016     0.0
2017     0.0
2018     0.0
2019     0.0
2020     0.6
2021    75.4
2022    83.2
2023    87.9
2024    78.1
2025    79.5
2026    88.4
2027    95.1
Name: Electric Range, dtype: float64

State
WA    299353
CA       174
VA       101
FL        47
TX        44
Name: count, dtype: int64 

Electric Vehicle Type
Battery Electric Vehicle (BEV)            241840
Plug-in Hybrid Electric Vehicle (PHEV)     58302
Name: count, dtype: int64 

52 makes


Make
TESLA         122291
CHEVROLET      20218
FORD           15997
NISSAN         15994
KIA            14897
TOYOTA         14374
HYUNDAI        12581
BMW            12395
RIVIAN         10150
VOLVO           8255
VOLKSWAGEN      7934
JEEP            6161
AUDI            6053
SUBARU          3693
CHRYSLER        3644
Name: count, dtype: int64

39 WA counties
Largest county share: 48.1% (King)


County
King         143976
Snohomish     37803
Pierce        25169
Clark         18897
Thurston      11043
Kitsap        10383
Spokane        8778
Whatcom        7573
Benton         4546
Skagit         3596
Name: count, dtype: int64

Rows with several utilities in one field: 224,514 (74.8%)


Data Summary

In [5]:
print(f"""
Total rows:            {len(df):,}
Duplicate vehicle IDs: {df['DOL Vehicle ID'].duplicated().sum():,}
Out-of-state rows:     {(df['State'] != 'WA').sum():,}
Missing location:      {df['Vehicle Location'].isna().sum():,}
Model years before 2011: {(df['Model Year'] < 2011).sum():,}
Range unknown (= 0):   {zero.mean() * 100:.1f}%
Largest county share:  {counties.iloc[0] / counties.sum() * 100:.1f}%
""")


Total rows:            300,142
Duplicate vehicle IDs: 0
Out-of-state rows:     789
Missing location:      24
Model years before 2011: 50
Range unknown (= 0):   65.7%
Largest county share:  48.1%



In the final use of the Dataset, the out of state rows will be removed along with the missing location as they don't fall in line with the focus on data from the state of washington. In the data analysis range will not be used as an overwhelming amount of observations have unknown range equaling 0 (65.7% of the data). Since there are no duplicates no changes need to be used there. Clean Alternative Fuel Vehicle (CAFV) Eligibility is also not utilized as it depends on range which is missing much data.

Final Cleaning Numbers done in python File

In [6]:
before = len(df)

# 1. Out-of-state rows: removed (focus is Washington)
out_of_state = (df["State"] != "WA").sum()
clean = df[df["State"] == "WA"].copy()

# 2. Missing location: removed (cannot be placed on the map)
no_location = clean["Vehicle Location"].isna().sum()
clean = clean.dropna(subset=["Vehicle Location"])

# 3. Duplicates: checked, none found, no change
dups = clean["DOL Vehicle ID"].duplicated().sum()

# 4. Electric Range and CAFV Eligibility: not used
zero_range = (df["Electric Range"] == 0).mean() * 100
clean = clean.drop(columns=["Electric Range",
                            "Clean Alternative Fuel Vehicle (CAFV) Eligibility"])

print(f"""Starting rows:              {before:,}
Out-of-state removed:       {out_of_state:,}
Missing location removed:   {no_location:,}
Duplicate vehicle IDs:      {dups:,} (no change)
Range unknown (= 0):        {zero_range:.1f}% -> Electric Range and CAFV Eligibility dropped
Final rows:                 {len(clean):,}""")

Starting rows:              300,142
Out-of-state removed:       789
Missing location removed:   9
Duplicate vehicle IDs:      0 (no change)
Range unknown (= 0):        65.7% -> Electric Range and CAFV Eligibility dropped
Final rows:                 299,344
